# CD1 · Aula 07 — Laboratório: validação cruzada aninhada (Nested CV)

A **busca em grade** (`GridSearchCV`) você viu hoje por cima: você dá uma grade de valores,
ela varre tudo com validação cruzada e devolve a melhor combinação. Ela ganha a aula
inteira na **próxima aula** — como montar a grade, quanto custa, o que fazer quando ela
explode. Aqui ela entra só como ferramenta: é o que o **laço interno** usa para escolher.

O assunto de hoje é o outro laço. A nota que a busca imprime (`best_score_`) é o **máximo**
de várias tentativas, e o máximo sempre pega um tanto de sorte — então ela não serve como
resposta para *"quanto esperar em dados novos?"*. Você vai ver a maldição do vencedor
acontecer com números, transformar a mesma busca em Nested CV com **uma linha**, refazer o
laço externo à mão para entender o que a linha faz, e medir o otimismo que o Nested remove.

Não é preciso dominar o `GridSearchCV` para fazer este laboratório: os exemplos mostram
tudo o que você precisa dele.

**Como usar.** Antes de cada exercício há uma célula de **exemplo**, já pronta e rodando,
com a mecânica num caso mínimo. Rode o exemplo, entenda, e só então resolva o exercício
logo abaixo. Rode o caderno **de cima para baixo**.

## Parte 0 — Ambiente e dados

In [ ]:
%matplotlib inline
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.model_selection import (GridSearchCV, cross_val_score,
                                     StratifiedKFold, train_test_split)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

TEAL, VERM, CINZA = "#0F766E", "#B91C1C", "#475569"

df = pd.read_csv("dados_aula07.csv")
X  = df.drop(columns="target").values
y  = df["target"].values

# a grade do dia: 3 valores de C x 3 de gamma = 9 combinacoes
GRADE = {"svc__C": [0.1, 1, 10], "svc__gamma": [0.01, 0.1, 1]}

def novo_pipe():
    """Um Pipeline NOVO a cada chamada: padroniza dentro de cada dobra, sem vazar."""
    return Pipeline([("esc", StandardScaler()), ("svc", SVC())])

interno = StratifiedKFold(5, shuffle=True, random_state=0)   # o laco que ESCOLHE
externo = StratifiedKFold(5, shuffle=True, random_state=0)   # o laco que MEDE

print("X:", X.shape, "| classes:", np.bincount(y))
print("grade:", GRADE)
print("combinacoes:", 3 * 3)

## Exercício 1 — A maldição do vencedor, sem modelo nenhum

Antes de qualquer `GridSearchCV`: o problema não é do scikit-learn, é de estatística. Simule tentativas que têm **todas o mesmo desempenho real 0,80**, com ruído de medição (desvio 0,04). Faça isso com **9** tentativas (o tamanho da nossa grade), **100** e **1000**, e monte uma tabela com a média e o máximo de cada caso.

Repare no que acontece com a média e o que acontece com o máximo quando o número de tentativas cresce.

**Exemplo antes de começar.** com **5** medições de um mesmo valor real 0,80, a média fica perto de 0,80 — mas o máximo já sai acima. Nenhuma das cinco é melhor que as outras: só uma delas teve sorte.

In [ ]:
rng = np.random.default_rng(0)
medicoes = 0.80 + rng.normal(0, 0.04, size=5)
print("as 5 medicoes:", np.round(medicoes, 3))
print(f"media  = {medicoes.mean():.3f}   <- perto do real (0,80)")
print(f"MAXIMO = {medicoes.max():.3f}   <- acima do real, so por sorte")

**Agora é com você.**

In [ ]:
# TODO — resolva aqui
rng = np.random.default_rng(0)
for n in [9, 100, 1000]:
    m = 0.80 + rng.normal(0, 0.04, size=n)
    print(...)

*Sua resposta:*

_(escreva aqui)_

## Exercício 2 — O que o GridSearchCV devolve

Rode a busca da `GRADE` completa sobre `novo_pipe()`, usando `cv=interno`, em **toda** a base. Imprima o `best_params_`, o `best_score_` e a tabela do `cv_results_` com as colunas `param_svc__C`, `param_svc__gamma`, `mean_test_score` e `std_test_score`, ordenada por `rank_test_score`.

**Exemplo antes de começar.** uma busca minúscula, com dois valores de `C` e nada mais, só para você ver o formato do que sai. São três coisas: a combinação vencedora (`best_params_`), a média de validação dela (`best_score_` — validação, não teste) e a tabela com uma linha por combinação (`cv_results_`).

In [ ]:
mini = GridSearchCV(novo_pipe(), {"svc__C": [0.1, 10]}, cv=interno)
mini.fit(X, y)
print("best_params_:", mini.best_params_)
print(f"best_score_ : {mini.best_score_:.4f}   (media das 5 dobras INTERNAS)")
print("\ncv_results_ tem uma linha por combinacao:")
print(pd.DataFrame(mini.cv_results_)[["param_svc__C", "mean_test_score"]].to_string(index=False))

**Agora é com você.**

In [ ]:
# TODO — resolva aqui
busca = GridSearchCV(novo_pipe(), ..., cv=...)
busca.fit(X, y)
print(...)

## Exercício 3 — O `best_score_` é o máximo daquela tabela

Mostre que `busca.best_score_` é exatamente o **máximo** da coluna `mean_test_score` — ou seja, o campeão de 9 tentativas, que é o caso `n = 9` do Exercício 1. Em seguida conte quantas das 9 combinações ficam a **menos de um desvio-padrão** da campeã (o *platô*).

**Exemplo antes de começar.** o `.max()` de uma coluna e uma comparação simples. Repare que o desvio entre dobras da campeã costuma ser **maior** que a distância dela para a segunda colocada.

In [ ]:
tab = pd.DataFrame(busca.cv_results_)
campea = tab.loc[tab.rank_test_score == 1].iloc[0]
print(f"media da campea : {campea.mean_test_score:.4f}")
print(f"desvio dela     : {campea.std_test_score:.4f}   <- entre as 5 dobras internas")
print(f"2o colocado     : {tab.mean_test_score.nlargest(2).iloc[1]:.4f}")
print(f"\ndistancia 1o -> 2o: {campea.mean_test_score - tab.mean_test_score.nlargest(2).iloc[1]:.4f}")

**Agora é com você.**

In [ ]:
# TODO — resolva aqui
print("best_score_ ==", ...)
dentro = ...
print("combinacoes dentro de 1 desvio da campea:", dentro, "de 9")

*Sua resposta:*

_(escreva aqui)_

## Exercício 4 — A linha que transforma a busca em Nested CV

Agora o ponto da aula. Passe o **objeto `busca`** (o `GridSearchCV` inteiro) como se fosse um modelo qualquer para o `cross_val_score`, com `cv=externo`. Imprima as 5 notas externas, a média, o desvio — e a diferença para o `best_score_` do Exercício 2.

**Exemplo antes de começar.** o `cross_val_score` aplicado a um modelo comum: ele devolve **uma nota por dobra**. É esse formato que vamos reaproveitar — só que no lugar do modelo vai a busca inteira.

In [ ]:
simples = cross_val_score(novo_pipe(), X, y, cv=externo)
print("um SVC comum, 5 dobras:", np.round(simples, 4))
print(f"media = {simples.mean():.4f}  |  desvio = {simples.std():.4f}")
print("\nO cross_val_score chama fit() no treino de cada dobra e pontua no teste dela.")

**Agora é com você.**

In [ ]:
# TODO — resolva aqui
notas = cross_val_score(..., X, y, cv=...)
print(...)

*Sua resposta:*

_(escreva aqui)_

## Exercício 5 — Quanto isso custou

O Nested não é de graça. Calcule o número de ajustes de modelo de cada abordagem (`combinações × dobras internas` para a busca sozinha; `combinações × internas × externas` para o Nested) e **meça o tempo** dos dois com `time.perf_counter()`. A razão entre os tempos deve ficar perto da razão entre as contas.

**Exemplo antes de começar.** como cronometrar um trecho. Aqui só a busca sozinha: 9 combinações × 5 dobras internas.

In [ ]:
import time
t0 = time.perf_counter()
GridSearchCV(novo_pipe(), GRADE, cv=interno).fit(X, y)
t_busca = time.perf_counter() - t0
print(f"9 combinacoes x 5 dobras = {9*5} ajustes  ->  {t_busca:.2f} s")

**Agora é com você.**

In [ ]:
# TODO — resolva aqui
import time
n_busca  = 9 * 5
n_nested = ...
t0 = time.perf_counter()
cross_val_score(...)
t_nested = time.perf_counter() - t0
print(...)

*Sua resposta:*

_(escreva aqui)_

## Exercício 6 — O laço externo à mão

Refaça o Exercício 4 **sem** o `cross_val_score`, para ver o que ele faz por dentro. Percorra `externo.split(X, y)`; em cada rodada, rode um `GridSearchCV` novo **só no treino** e pontue **no teste externo**. Imprima, por rodada, o `best_params_` escolhido e a nota. No fim, compare a média com a do Exercício 4.

**Exemplo antes de começar.** como um `StratifiedKFold` entrega os índices. Cada rodada devolve dois vetores de posições: o treino e o teste daquela dobra.

In [ ]:
for i, (tr, te) in enumerate(StratifiedKFold(3, shuffle=True, random_state=0).split(X, y), 1):
    print(f"rodada {i}: treino {len(tr)} exemplos | teste {len(te)} exemplos"
          f" | positivos no teste: {y[te].sum()}")

**Agora é com você.**

In [ ]:
# TODO — resolva aqui
notas_mao = []
for i, (tr, te) in enumerate(externo.split(X, y), 1):
    g = GridSearchCV(...).fit(X[tr], y[tr])
    nota = ...
    notas_mao.append(nota)
    print(...)

*Sua resposta:*

_(escreva aqui)_

## Exercício 7 — O hiperparâmetro escolhido varia — e está tudo bem

Olhe a lista `escolhas` do exercício anterior. Conte quantas combinações **distintas** apareceram nas 5 rodadas e mostre a frequência de cada uma. Depois responda, no campo de texto: **qual é "o" melhor `C`?**

**Exemplo antes de começar.** como contar as repetições de uma lista de tuplas com o `Counter`.

In [ ]:
from collections import Counter
exemplo = [("a", 1), ("b", 2), ("a", 1), ("a", 1)]
print(Counter(exemplo))
print("distintas:", len(set(exemplo)))

**Agora é com você.**

In [ ]:
# TODO — resolva aqui
from collections import Counter
print(...)

*Sua resposta:*

_(escreva aqui)_

## Exercício 8 — O otimismo, medido em 10 sementes

Uma rodada só pode ter sido sorte. Para `semente` de 0 a 9, monte um `interno` e um `externo` com aquela semente, e colete duas coisas: o `best_score_` da busca (**não-aninhado**) e a média do `cross_val_score` (**aninhado**). Faça um gráfico com as duas curvas e reporte o otimismo médio e em quantas das 10 sementes o Nested ficou **abaixo**.

**Exemplo antes de começar.** a mesma coleta, com 2 sementes só, para você ver o formato do laço antes de rodar as 10 (o exercício leva alguns segundos).

In [ ]:
for s in [0, 1]:
    i_cv = StratifiedKFold(5, shuffle=True, random_state=s)
    o_cv = StratifiedKFold(5, shuffle=True, random_state=s)
    g = GridSearchCV(novo_pipe(), GRADE, cv=i_cv).fit(X, y)
    a = cross_val_score(g, X, y, cv=o_cv).mean()
    print(f"semente {s}: best_score_ = {g.best_score_:.4f} | nested = {a:.4f}")

**Agora é com você.**

In [ ]:
# TODO — resolva aqui
nao_aninhado, aninhado = [], []
for s in range(10):
    ...
# grafico

*Sua resposta:*

_(escreva aqui)_

## Exercício 9 — Por que não basta um teste separado

O caminho mais comum — separar 30% para teste, buscar no resto, medir uma vez — também respeita a regra de ouro. O problema é outro: o número depende do **corte**. Repita esse caminho com 10 sementes diferentes de `train_test_split` e guarde as 10 notas de teste. Plote-as junto com a faixa do Nested CV (média ± desvio do Exercício 4) e compare a **amplitude** (máximo − mínimo) com o desvio do Nested.

**Exemplo antes de começar.** um hold-out só, com a semente 0: separa 30%, busca no treino, mede uma vez no teste. É o fluxo que você vai usar na próxima aula, quando a busca em grade for o assunto principal.

In [ ]:
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)
g = GridSearchCV(novo_pipe(), GRADE, cv=5).fit(X_tr, y_tr)
print("escolheu:", g.best_params_)
print(f"best_score_ (validacao): {g.best_score_:.4f}")
print(f"TESTE guardado, 1 vez  : {g.score(X_te, y_te):.4f}")

**Agora é com você.**

In [ ]:
# TODO — resolva aqui
testes = []
for s in range(10):
    ...
# grafico + amplitude

*Sua resposta:*

_(escreva aqui)_

## Exercício 10 — O número e o modelo

Feche o ciclo. O Nested CV já lhe deu o **número**; ele não dá o **modelo**. Rode uma única busca em **toda** a base para obter a combinação que vai para produção, e imprima as duas coisas lado a lado: o que você anuncia e o que você entrega.

**Exemplo antes de começar.** a busca final não tem nada de especial — é o mesmo `GridSearchCV` do Exercício 2, rodado em todos os dados. O que muda é o papel dela: aqui ela não mede nada, só escolhe.

In [ ]:
final = GridSearchCV(novo_pipe(), GRADE, cv=interno).fit(X, y)
print("best_params_ :", final.best_params_)
print("best_estimator_ ja vem treinado:", final.best_estimator_.named_steps["svc"])

**Agora é com você.**

In [ ]:
# TODO — resolva aqui
final = ...
print("ANUNCIO  :", ...)
print("ENTREGO  :", ...)

*Sua resposta:*

_(escreva aqui)_

---
## Fecho

Em uma frase: **quem escolhe não mede**.

| | o que faz | o que devolve |
|---|---|---|
| `GridSearchCV` | varre a grade com CV dentro do treino | `best_params_`, `best_estimator_` — **o modelo** |
| `cross_val_score(GridSearchCV(...))` | repete a busca inteira em K dobras externas | K notas → média ± desvio — **o número** |

O `best_score_` não é nenhum dos dois: é o máximo de N tentativas na validação, e o
Exercício 1 mostrou que esse máximo cresce com N. Ele serve para **escolher**, nunca
para reportar.

**Quando cada um basta.** Se você tem uma base grande e um teste separado que ninguém
nunca tocou, a busca + uma medição no teste já resolve — e por um quinto do custo
(Exercício 5). O Nested CV ganha quando a base é pequena, quando reservar um teste fixo
dói, ou quando aquele teste já foi olhado alguma vez e deixou de ser imparcial.

**Na próxima aula** a busca em grade vira o assunto principal: como escolher os valores da
grade, o custo combinatório, o `scoring` certo e o que fazer quando a grade explode. O que
você fixou aqui — *quem escolhe não mede* — é a régua que vai valer lá também.